# Instructions
Before you start this lesson please click the "Run all" button at the top of the page:

![Screenshot showing run all button](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/run-all-button.png?raw=true)

This might take about a minute or more.

In [ ]:
from io import BytesIO

import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
import cv2

# Images and Convolutions
In this lesson we're going to learn how we can take images and use them in neural networks.

Computers only know how to do things with numbers. In our previous data modeling activity we took concepts related to weather and represented them as numbers:

- Humidity: A percentage from 0% to 100%
- Temperature: A number in Fahrenheit
- Cloud Coverage: A percentage of how much of the sky is obscured by clouds from 0% to 100%

These weather measurements are easily translated into numbers. But how do we take something like images and turn those into numbers?

## Pixels
To learn how computers "see" images as numbers we need to learn about pixels.

To a computer an image is a series of squares which are each one color. We call these squares pixels.

You're probably already familiar with pixels even if you haven't heard the term. When a photo or video is really low quality can see each individual pixel.

Like in this photo:

<img alt="Low resolution photo of a cat" src="https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-screenshot.png?raw=true" width="400px" />

Look at the ears and face of the cat. You can clearly see boxes which are each one color (pixels).

You can think of an image then as a list of pixels.

## Colors
So if an image is made up of a bunch of pixels. And each pixel is an individual color. Then how do we represent colors as numbers?

Colors can be represented by mixing the primary colors red, green, and blue together in different amounts. If you've taken an art class then this may be familiar to you.

So to represent colors in computers we just keep track of how much red, blue, and green are in the color.

Play with the demo below to see how you can make colors out of combinations of red, green, and blue:

In [ ]:
# @title
def update_color(r, g, b):
    color_hex = f'#{r:02x}{g:02x}{b:02x}'
    display(HTML(f"<div style='width:100px; height:100px; background-color:{color_hex}; border: 1px solid black;'></div>"))

red_slider = widgets.IntSlider(min=0, max=255, step=1, value=113, description='Red:')
green_slider = widgets.IntSlider(min=0, max=255, step=1, value=60, description='Green:')
blue_slider = widgets.IntSlider(min=0, max=255, step=1, value=255, description='Blue:')


output = widgets.interactive_output(update_color, {'r': red_slider, 'g': green_slider, 'b': blue_slider})

display(red_slider, green_slider, blue_slider, output)

In computers we track how much of each primary color is present using a number between 0 and 255 (0 meaning none of the color, 255 meaning all of the color).

> **Why 0 to 255?**  
> This number is convenient for computers to store. As 255 is the biggest number that can be represented with 8 ones and zeros.

In order to store this data we use a list. Where the:

- First item is the amount of red
- Second item is the amount of blue
- Third item is the amount of green

So for example:

In [ ]:
# [red, green, blue]
color = [113, 60, 255]

This list above represents a blue-ish purple color.

## Images as Pixels with Colors
So far we've learned that:

- An image is just a list of pixels
- Each pixel is a color
- Each color is just a list of the amount of red, green, and blue

Then an image is really a list of colors. For example this could represent an image:

In [ ]:
image = [
    [0, 34, 87],
    [90, 46, 1],
    [0, 0, 0],
    [89, 30, 111],
]

In [ ]:
# @title
# Convert the list of lists (RGB pixels) to a NumPy array of type uint8
image_array = np.array(image, dtype=np.uint8)

# Reshape the array to (height, width, channels) for a horizontal strip of pixels
# Here, height is 1, width is the number of pixels (len(image)), and channels is 3 (RGB)
display_image_array = image_array.reshape(1, len(image), 3)

# Display the image using matplotlib
plt.imshow(display_image_array)
plt.axis('off') # Hide axes for a cleaner image display
plt.show()

# Check In - Images
We just learned how images are converted into numbers. Here are the things you should know:

- Images are made of up a list of pixels
- Each pixel is square which is one color
- Colors are made by mixing the primary colors: red, green, and blue
- Colors are represented as numbers by the amount of red, green, and blue in the color
- Use numbers between 0 and 255

# Understanding Images as a Computer
So now we know how images are represented as numbers. But how do computers "look" at those images and understand what is in an image?

- A first approach might be to just let the computer read all the numbers for each pixel
- But that's a ton of data
- And doesn't really match how we as humans understand images...

## How do Humans Understand Images?
Before we learn how computers understand images, let's think about how humans understand images.

When you look at a photo what do you pick out?

- Corners?
- Lines?
- Objects?
- Contrast between light and dark areas?

When humans look use their eyes we aren't looking at each individual pixel on its own. We're looking at patterns in the image. We're picking out features of objects.

## Transfering this to Computers
Okay, so if humans don't look at individual pixels, but instead pick up on patterns in the image. Then how can computers do this?

For computers to pick up on patterns in images we use something called "convolutions". This might sound like a crazy convoluted word. But, you probably have used convolutions in your daily life before...

# Convolutions
Convolutions are the same technology behind photo filters like those in Instagram, snapchat, and others. As well image editing tools like Photoshop.

For example, here is that cat photo from earlier but run through a bunch of different types of convolutions:


In [ ]:
# @title
response = requests.get("https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/medium-resolution-photo.jpg?raw=true")
img = np.array(Image.open(BytesIO(response.content)))

def blur(src):
  kernel = np.array([
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
  ])

  return cv2.filter2D(src, -1, kernel)

def sharpen(src):
  kernel = np.array([
      [0, -1, 0],
      [-1, 5, -1],
      [0, -1, 0]
  ])

  return cv2.filter2D(src, -1, kernel)

def emboss(src):
  kernel = np.array([
      [-2, -1, 0],
      [-1, 1, 1],
      [0, 1, 2]
  ])

  return cv2.filter2D(src, -1, kernel)


def sepia(src):
    kernel = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def blue(src):
    kernel = np.array([
         [0.25, 0.4, 0.15],
         [0.35, 0.65, 0.25],
         [0.4, 0.75, 0.35],
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def left_sobel(src):
    kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ])
    return cv2.filter2D(src, -1, kernel)

def outline(src):
    kernel = np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1],
    ])
    return cv2.filter2D(src, -1, kernel)

def show_images(before, after, name):
    plt.figure(figsize=(14, 6), dpi=100)
    plt.subplot(1, 2, 1)
    plt.title("Original")
    plt.imshow(np.array(before))
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(name)
    plt.imshow(np.array(after))
    plt.axis('off')

    plt.show()


show_images(img, blur(img), "Blurred")
show_images(img, sharpen(img), "Sharpened")
show_images(img, emboss(img), "Embossed")
show_images(img, sepia(img), "Sepia")
show_images(img, blue(img), "Blue")
show_images(img, left_sobel(img), "Left Sobel")
show_images(img, outline(img), "Outline")

So a convolution changes how an image looks. But what exactly is a convolution?

## Step by Step
A convolution changes each pixel in our image by a little bit.

To do this we:

1. Arrange the pixels in our image in an X, Y grid like so:
  ![Screenshot of cat super zoomed in](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-super-zoomed-screenshot.png?raw=true)
2. For each pixel we look at its direct neighbors:  
  _(Target pixel we are trying to change highlighted in blue)_  
  _(Surrounding neighbor pixels highlighted in red)_  
  ![Target pixel highlighted in blue, neighbor pixels highlighted in red](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-super-zoomed-screenshot-kernel-area.png?raw=true)
3. Get the new value of the pixel by:
  - Multiply each neighbor pixel's color by a number we define
    - Remember: A pixel's color is defined by 3 numbers
    - The amount of red, green, and blue
    - Ranging from 0 to 255
    - So we just multiply each of these numbers to get the new red, green, and blue color
  - Add up the multiplied values together
  - Use this new value as the center pixel's new color

We complete this process for every pixel in the image. Getting its new color by multiplying and adding up the color values of their neighboring pixels.

## Kernels
So how are we able to get so many different effects just by completing this same process? The answer lies in which number we multiply the color values by.

- We can multiply each neighboring pixel by a different value based on its position
- We can multiply neighboring pixels by different amounts
- We can also multiply the target pixel by a number and include that

These numbers that we choose are sort of the like the paint brush type our convolution uses to paint the new pixel colors. We call this paint brush a **kernel**.

Let's look at an example kernel to get a better sense of them:

![Image of sobel kernel numbers](https://github.com/catvec/machine-learning-camp/blob/main/assets/images-and-convolutions/low-resolution-photo-super-zoomed-screenshot-kernel-numbers.png?raw=true)

To get the new color of the pixel in blue we multiply each pixel and add them up:

- First / top row
  - Left by -1
  - Center by 0
  - Right by 1
- Second / middle row
  - Left by -2
  - Center by 0
  - Right by 2
- Third / last row
  - Left by -1
  - Center by 0
  - Right by 1

That's a lot to write out. So what we do is record our kernel in a list. Where each item in the list indicates what number we multiply that row of pixels by. The same kernel as above written as a list is:

```python
kernel = [
  [-1, 0, 1],
  [-2, 0, 2],
  [-1, 0, 1]
]
```

The kernels for each of the effects in the photos above are:

In [ ]:
blur = [
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
]

sharpen = [
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
]

emboss = [
    [-2, -1, 0],
    [-1, 1, 1],
    [0, 1, 2]
]

sepia = [
    [0.393, 0.769, 0.189],
    [0.349, 0.686, 0.168],
    [0.272, 0.534, 0.131]
]

blue = [
    [0.25, 0.4, 0.15],
    [0.35, 0.65, 0.25],
    [0.4, 0.75, 0.35],
]

left_sobel = [
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
]

outline = [
    [-1, -1, -1],
    [-1, 8, -1],
    [-1, -1, -1],
]

# Your Turn - Kernels
Try making your own kernels. Pick different values for each neighbor pixel in the kernel and see how it affects the result.

1. Define your kernel in the `kernel` variable
  - Remember the first array is for the first row, second array for second row, third array for third row
  - The first item of each array is the left most pixel, the second for the second pixel, and the third for the last pixel
2. Run the code cell to see how the image is affected

Try making at least 5 different kernels.

In [ ]:
# Define your kernel here
kernel = [
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
]

# Don't touch this
# (This code applies your kernel using a convolution)
after = cv2.filter2D(img, -1, np.array(kernel))
after = np.clip(after, 0, 255).astype(np.uint8)

show_images(img, after, "After")

# Check In - Convolutions
We just learned that convolutions are ways to apply effects to an image. Here are the things you should know:

- A convolution sets new color values for each pixel in an image
- This is done by looking at the neighboring pixels of each pixel
- Each neighboring pixel is multiplied by a value from a kernel, then added up, and used as the new pixel color
- The kernel influences what effect the convolution has on the image

# Use of Convolutions
So we can make some pretty funky looking photos with convolutions. But how does that help a computer understand what is in an image?

Well we can pick specific kernels which make the resulting image show some useful things:

- Edges of objects
- Overall shape of objects
- Centers of objects

For example:

In [ ]:
# @title
def blur(src):
  kernel = np.array([
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
      [1/9, 1/9, 1/9],
  ])

  return cv2.filter2D(src, -1, kernel)

def sharpen(src):
  kernel = np.array([
      [0, -1, 0],
      [-1, 5, -1],
      [0, -1, 0]
  ])

  return cv2.filter2D(src, -1, kernel)

def emboss(src):
  kernel = np.array([
      [-2, -1, 0],
      [-1, 1, 1],
      [0, 1, 2]
  ])

  return cv2.filter2D(src, -1, kernel)


def sepia(src):
    kernel = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def blue(src):
    kernel = np.array([
         [0.25, 0.4, 0.15],
         [0.35, 0.65, 0.25],
         [0.4, 0.75, 0.35],
    ])
    after = cv2.transform(src, kernel)
    return np.clip(after, 0, 255).astype(np.uint8)

def left_sobel(src):
    kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ])
    return cv2.filter2D(src, -1, kernel)

def outline(src):
    kernel = np.array([
        [-1, -1, -1],
        [-1, 8, -1],
        [-1, -1, -1],
    ])
    return cv2.filter2D(src, -1, kernel)

blurred_img = img
for _i in range(100):
    blurred_img = blur(blurred_img)

show_images(img, blurred_img, "Blurred")
show_images(img, left_sobel(img), "Edges (Sobel)")
show_images(img, outline(img), "Edges (Outline)")

- Blurred
  - With the blur kernel the computer can't see any fine details
  - This is ideal for detecting the general gist of where large objects are
  - For humans this is sort of like when you just glance down the road and see if there are any cars
    - You don't look to see the license plate of the car or the model of the car
    - You just see if there is a car or how far away it is
- Edges
  - There are 2 different examples of how to detect edges (Sobel and outline)
  - The outline kernels remove all finer detail about what is happening inside objects
  - Instead only presents the borders of the objects
  - Good for detecting what an object is based off its shape
  - For humans this is sort of like when you look at the sky and see the silhouette of a plane
    - You are really only looking at the outline of the object in the sky to see what it is

# Convolutions in Neural Networks
So how do convolutions fit into neural networks?

- A neural network will take an input image and run it through several convolutions using many different kernels
- The output of those convolutions will be used as the input to the first layer of neurons

If a neural network runs its input through convolutions we call it a **convolutional neural network**.

These convolutions allow the neural network to learn what objects are. The output of convolutions is run through several hidden neural network layers. Each layer learns a slightly more complex representation of an object. Starting from just pixels, and resulting in complete identification of objects:

- Input image is run through several different convolutions
- **First layer:** Receives output of convolutions
- **Second layer:** Uses convolutions to learn simple patterns
- **Several middle layers:** Can recognize object parts (wheels, eyes, feathers)
- **Final layer:** Can recognize complete objects

Notice how the layers build on each other. The second layer can only recognize simple patterns. But the middle layers then use those patterns to recognize parts. And then the final layer uses those parts to recognize complete objects.

# Check In - Convolutions in Neural Networks
We just learned how convolutions are used in neural networks. Here are the key points you should know:

- The input to a neural network is sent through multiple different convolutions
- The output of these convolutions is sent to the first layer of neurons
- Multiple hidden layers of neurons use the output of these convolutions to build an understanding of what is in the image

# Convolutions and Training
How does someone creating a neural network know which kernels to use in the input convolutions?

- We could just pick some useful kernels like the ones we saw above
- But what if that isn't exactly what the neural network needs based on the training data
  - For example if the neural network is supposed to recognize airplanes we might need several really good kernels which distinguish edges
  - But if neural network is supposed to recognize red balls we may need a kernel which can distinguish circles really well

So we can't necessarily use the same useful kernels we saw above for all purposes. Instead what we do is let the neural network learn which kernels are most useful for its task.

During training we:

1. Start with random kernels
2. After each round of training we use the error and derivatives to determine how we can change the kernel to get a better result (lower error)

The kernels are essentially just weights in a different form! Remember, neuron inputs are multiplied by weights. This is very similar to kernels where neighbor pixels are multiplied by a number from the kernel! So we can use the same training techniques to find the best values for our kernels.

## The Math of Convolutions
> **Bonus:**  
> Don't get overwhelmed by the equations below, this is just explaining how we can use the techniques we already know (getting the derivative of $y = m \cdot x + b$) to figure out how to update kernels.
>
> If you want, ignore this math and just make sure you understand that we **can** update kernels to reduce the error (make our neural network better)

The idea that we can do the same type of updating of a kernel might seem a bit counterintuitive at first.

Imagine we have an image with only 9 pixels:

- We will label these pixels $p_1$, $p_2$, $p_3$, ..., $p_9$
- To get the red, green, or blue value for a pixel we will say $p_{1_\text{red}}$, $p_{1_\text{green}}$, $p_{1_\text{blue}}$, ect
- The values of our kernel will be labeled similar to our pixels, except we will use the letter $k$: $k_1$, $k_2$, $k_3$, ..., $k_9$

The math equation to find a new red pixel value would be:  

$\text{new red} = (p_{1_\text{red}} \cdot k_1) + (p_{2_\text{red}} \cdot k_2) + (p_{3_\text{red}} \cdot k_3) + (p_{4_\text{red}} \cdot k_4) + (p_{5_\text{red}} \cdot k_5) + (p_{6_\text{red}} \cdot k_6) + (p_{7_\text{red}} \cdot k_7) + (p_{8_\text{red}} \cdot k_8) + (p_{9_\text{red}} \cdot k_9)$

Or if you want to use a fancy, but lazier notation:

$\sum^9_{i=1}{p_{i_\text{red}} \cdot k_i}$

We would repeat this 2 more times for green and blue as well. Then we would repeat this equation for every single pixel in the input image.

- Although this would result in a ton of equations
- It is still an equation we can find the derivative of!
- So we can figure out how to change each $k_1$, $k_2$, $k_3$, ..., $k_9$ to reduce the error

# Check In - Training Kernels
We just learned how you can use neural network training techniques to find the best kernels for a neural network. Here are the key points you should understand:

- Different kernels work better for different things
- So we can't use the same set of handy kernels for every neural network and dataset
- Instead we use the same training techniques to find which kernels allow the neural network to be the most accurate